# C3-gradient-descent — Practice p19 — Solution

**Type:** scenario analysis · **Difficulty:** advanced · **Concepts:** gradient-descent, learning-rate, mse-loss, broadcasting, aggregation-axis

*Reasoning is required. Coding is limited to committing five typed diagnosis strings and then running the marked verifier.*

Five teams intended to run mean-MSE gradient descent. Each transcript contains exactly one planted bug. Run the transcript cell unchanged, diagnose from its printed trace, and write all five diagnoses on paper **before** editing the answer cell. Use each canonical diagnosis exactly once:

- `"eta-too-large"`
- `"sum-vs-mean"`
- `"gradient-sign-flip"`
- `"un-zeroed-accumulator"`
- `"wrong-axis-broadcast"`

The fields are: losses after successive steps; gradient norm; $\nabla L\cdot\Delta\theta$ (negative for a descent-direction update); observed and intended residual shapes; and, for a duplicated-batch probe, the number of copies and the measured gradient-norm ratio. Explain each diagnosis in one sentence citing a trace field, then commit the exact typed identifiers `diagnosis_a` through `diagnosis_e`.

**Banned (zero points): editing the transcript artifact, running or reading the verifier before all five diagnoses are committed, assigning any expected/verdict value yourself, or replacing the required named variables with one anonymous list.**


In [1]:
transcripts = [
    {"name": "A", "loss": [3.84, 9.27, 24.61, 69.84, 205.17],
     "grad_norm": [4.1, 7.2, 12.5, 22.0, 40.3],
     "step_dot_grad": [-0.81, -2.49, -7.66, -23.8, -74.2],
     "residual_shape": (12,), "expected_residual_shape": (12,),
     "copies": 1, "grad_ratio": 1.0},
    {"name": "B", "loss": [5.12, 4.43, 3.91, 3.56, 3.34],
     "grad_norm": [18.6, 15.1, 12.4, 10.3, 8.7],
     "step_dot_grad": [-1.73, -1.14, -0.77, -0.53, -0.38],
     "residual_shape": (12,), "expected_residual_shape": (12,),
     "copies": 5, "grad_ratio": 5.0},
    {"name": "C", "loss": [2.76, 3.04, 3.37, 3.75, 4.19],
     "grad_norm": [2.8, 3.0, 3.2, 3.4, 3.7],
     "step_dot_grad": [0.39, 0.45, 0.51, 0.58, 0.68],
     "residual_shape": (12,), "expected_residual_shape": (12,),
     "copies": 1, "grad_ratio": 1.0},
    {"name": "D", "loss": [6.31, 4.82, 3.15, 2.94, 5.88],
     "grad_norm": [3.2, 5.9, 8.7, 11.8, 15.2],
     "step_dot_grad": [-0.72, -1.92, -3.86, -7.14, -11.9],
     "residual_shape": (12,), "expected_residual_shape": (12,),
     "copies": 1, "grad_ratio": 1.0},
    {"name": "E", "loss": [2.24, 1.71, 1.53, 1.49, 1.48],
     "grad_norm": [2.2, 1.5, 1.0, 0.7, 0.5],
     "step_dot_grad": [-0.24, -0.12, -0.06, -0.03, -0.02],
     "residual_shape": (12, 12), "expected_residual_shape": (12,),
     "copies": 1, "grad_ratio": 1.0},
]

for trace in transcripts:
    print(trace)



{'name': 'A', 'loss': [3.84, 9.27, 24.61, 69.84, 205.17], 'grad_norm': [4.1, 7.2, 12.5, 22.0, 40.3], 'step_dot_grad': [-0.81, -2.49, -7.66, -23.8, -74.2], 'residual_shape': (12,), 'expected_residual_shape': (12,), 'copies': 1, 'grad_ratio': 1.0}
{'name': 'B', 'loss': [5.12, 4.43, 3.91, 3.56, 3.34], 'grad_norm': [18.6, 15.1, 12.4, 10.3, 8.7], 'step_dot_grad': [-1.73, -1.14, -0.77, -0.53, -0.38], 'residual_shape': (12,), 'expected_residual_shape': (12,), 'copies': 5, 'grad_ratio': 5.0}
{'name': 'C', 'loss': [2.76, 3.04, 3.37, 3.75, 4.19], 'grad_norm': [2.8, 3.0, 3.2, 3.4, 3.7], 'step_dot_grad': [0.39, 0.45, 0.51, 0.58, 0.68], 'residual_shape': (12,), 'expected_residual_shape': (12,), 'copies': 1, 'grad_ratio': 1.0}
{'name': 'D', 'loss': [6.31, 4.82, 3.15, 2.94, 5.88], 'grad_norm': [3.2, 5.9, 8.7, 11.8, 15.2], 'step_dot_grad': [-0.72, -1.92, -3.86, -7.14, -11.9], 'residual_shape': (12,), 'expected_residual_shape': (12,), 'copies': 1, 'grad_ratio': 1.0}
{'name': 'E', 'loss': [2.24, 1.71, 1

- **A:** Every update is a descent direction (`step_dot_grad` stays negative), yet loss explodes from $3.84$ to $205.17$ while the gradient norm grows, so the learning rate is too large.
- **B:** Duplicating the batch five times multiplies the gradient norm by exactly $5$, revealing a sum where mean-MSE requires averaging.
- **C:** Every `step_dot_grad` is positive, so the update follows rather than opposes the gradient: its sign is flipped.
- **D:** The gradient norm rises every step and the loss reverses from $2.94$ to $5.88$, the accumulation signature of gradients carried across steps without being zeroed.
- **E:** The residual has shape `(12, 12)` instead of `(12,)`, directly identifying a broadcast along the wrong axis.


In [2]:
diagnosis_a: str = "eta-too-large"
diagnosis_b: str = "sum-vs-mean"
diagnosis_c: str = "gradient-sign-flip"
diagnosis_d: str = "un-zeroed-accumulator"
diagnosis_e: str = "wrong-axis-broadcast"

committed_diagnoses: list[str] = [
    diagnosis_a, diagnosis_b, diagnosis_c, diagnosis_d, diagnosis_e
]



### MARKED VERIFICATION CELL

Run this only after all five named answers are committed. It derives the grading codes from the actual `transcripts` artifact and reports agreement only; it never prints expected diagnoses.


In [3]:
import numpy as np

expected_count = len(transcripts)
if len(committed_diagnoses) != expected_count:
    raise RuntimeError(
        f"committed collection has length {len(committed_diagnoses)}; expected {expected_count}"
    )
if any(not isinstance(value, str) or not value.strip() for value in committed_diagnoses):
    raise RuntimeError("all five named diagnosis strings must be committed before verification")

diagnosis_codes = {
    "eta-too-large": 0,
    "sum-vs-mean": 1,
    "gradient-sign-flip": 2,
    "un-zeroed-accumulator": 3,
    "wrong-axis-broadcast": 4,
}
if any(value not in diagnosis_codes for value in committed_diagnoses):
    raise RuntimeError("each committed diagnosis must use one of the five canonical strings")

def derive_fault_code(trace):
    if tuple(trace["residual_shape"]) != tuple(trace["expected_residual_shape"]):
        return 4
    if all(value > 0 for value in trace["step_dot_grad"]):
        return 2
    if (
        trace["copies"] > 1
        and np.isclose(
            trace["grad_ratio"], float(trace["copies"]), atol=1e-12, rtol=0
        )
    ):
        return 1
    if (
        trace["loss"][-1] > 20 * trace["loss"][0]
        and all(value < 0 for value in trace["step_dot_grad"])
    ):
        return 0
    if (
        all(b > a for a, b in zip(trace["grad_norm"], trace["grad_norm"][1:]))
        and trace["loss"][-1] > 1.5 * trace["loss"][-2]
    ):
        return 3
    raise RuntimeError("transcript does not exhibit exactly one recognized fault signature")

derived_codes = [derive_fault_code(trace) for trace in transcripts]
committed_codes = [diagnosis_codes[value] for value in committed_diagnoses]
agreement = [hand == derived for hand, derived in zip(committed_codes, derived_codes)]
print("agreement with the five transcript-derived diagnoses:", agreement)
print("all_agree:", all(agreement))



agreement with the five transcript-derived diagnoses: [True, True, True, True, True]
all_agree: True


### Answer check


In [4]:
assert len(committed_diagnoses) == len(transcripts) == 5
assert all(type(value) is str and bool(value.strip()) for value in committed_diagnoses)
assert all(value in diagnosis_codes for value in committed_diagnoses)
assert len(set(committed_diagnoses)) == 5
assert all(agreement)
